In [ ]:
import niquests as requests
import base64
from PIL import Image
import io
import os

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
API_URL = "https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-image:generateContent"

prompt = "Redraw this portrait in the style of Jojo's Bizarre Adventure, exaggerated poses and vibrant colors with thick lines."

image_path = "/Users/clang88/Pictures/Screenshots/WhatsApp Image 2025-06-20 at 21.41.10.jpeg"  # Update this path as needed
with open(image_path, "rb") as img_file:
    img_bytes = img_file.read()
    # Optionally convert to JPEG if not already
    try:
        img = Image.open(io.BytesIO(img_bytes))
        if img.format != "JPEG":
            buf = io.BytesIO()
            img.convert("RGB").save(buf, format="JPEG")
            img_bytes = buf.getvalue()
    except Exception:
        pass
    img_b64 = base64.b64encode(img_bytes).decode("utf-8")

headers = {
    "x-goog-api-key": GEMINI_API_KEY,
    "Content-Type": "application/json",
}

payload = {
    "contents": [
        {
            "parts": [
                {"text": prompt},
                {
                    "inline_data": {
                        "mime_type": "image/jpeg",
                        "data": img_b64,
                    }
                },
            ]
        }
    ]
}

response = requests.post(API_URL, headers=headers, json=payload)
response.raise_for_status()
result = response.json()

for part in result.get("candidates", [{}])[0].get("content", {}).get("parts", []):
    if "text" in part:
        print(part["text"])
    elif "inline_data" in part:
        img_data = base64.b64decode(part["inline_data"]["data"])
        with open("generated_image.png", "wb") as out_img:
            out_img.write(img_data)